In [1]:
import os
import ssl
import random
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import certifi
    ssl._create_default_https_context = lambda: ssl.create_default_context(
        cafile=certifi.where()
    )
except ImportError:
    pass

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.models import EfficientNet_V2_S_Weights
from PIL import Image
from tqdm.auto import tqdm

In [2]:
BASE_DIR   = Path("ucsc-cse-144-spring-2026-final-project")
TRAIN_DIR  = BASE_DIR / "train"
TEST_DIR   = BASE_DIR / "test"
SAMPLE_CSV = BASE_DIR / "sample_submission.csv"
CKPT_DIR   = Path("checkpoints")
CKPT_PATH  = CKPT_DIR / "best_model.pt"
OUTPUT_CSV = Path("submission.csv")

CKPT_DIR.mkdir(exist_ok=True)

SEED = 43

IMG_SIZE = 300
BATCH_SIZE = 12
NUM_CLASSES = 100
NUM_WORKERS = 0

PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 60

PHASE1_LR = 8e-4
PHASE2_LR = 6e-5

WEIGHT_DECAY = 7e-4
LABEL_SMOOTH = 0.03
MIXUP_ALPHA = 0.0

PATIENCE = 14

In [3]:
def set_seed(seed: int = SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed()

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device} | torch {torch.__version__}")

Using device: mps | torch 2.8.0


In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TRAIN_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.88, 1.14)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

EVAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [5]:
class TrainDataset(Dataset):
    def __init__(self, root: Path, transform=None):
        self.transform = transform
        self.samples = []

        class_dirs = sorted(
            (p for p in root.iterdir() if p.is_dir() and p.name.isdigit()),
            key=lambda p: int(p.name),
        )

        for cls_dir in class_dirs:
            label = int(cls_dir.name)
            for img_path in sorted(cls_dir.glob("*.jpg")):
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


class TestDataset(Dataset):
    def __init__(self, test_dir: Path, file_ids: list[str], transform=None):
        self.test_dir = test_dir
        self.file_ids = file_ids
        self.transform = transform

    def __len__(self):
        return len(self.file_ids)

    def __getitem__(self, idx):
        fname = self.file_ids[idx]
        img = Image.open(self.test_dir / fname).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, fname

In [6]:
def stratified_split_indices(samples, val_per_class: int = 1, seed: int = SEED):
    rng = random.Random(seed)
    by_label = {}

    for idx, (_, label) in enumerate(samples):
        by_label.setdefault(label, []).append(idx)

    train_idx, val_idx = [], []

    for label in sorted(by_label):
        indices = by_label[label][:]
        rng.shuffle(indices)

        n_val = min(val_per_class, max(1, len(indices) // 5))
        val_idx.extend(indices[:n_val])
        train_idx.extend(indices[n_val:])

    return train_idx, val_idx

In [7]:
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)

    in_features = model.classifier[1].in_features

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 512),
        nn.SiLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )

    return model


def freeze_backbone(model: nn.Module):
    for name, p in model.named_parameters():
        p.requires_grad = "classifier" in name


def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True

def unfreeze_classifier_and_last_block(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False

    for p in model.features[-1].parameters():
        p.requires_grad = True

    for p in model.classifier.parameters():
        p.requires_grad = True

In [8]:
def mixup_data(x, y, alpha: float = MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)

    mixed_x = lam * x + (1.0 - lam) * x[idx]

    return mixed_x, y, y[idx], lam


def mixup_criterion(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, scheduler=None, use_mixup=True):
    model.train()

    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in tqdm(loader, leave=False, desc="train"):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        if use_mixup:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels)
            logits = model(imgs)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
        else:
            logits = model(imgs)
            loss = criterion(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        logits = model(imgs)
        loss = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

In [10]:
TTA_TF_LIST = [
    EVAL_TF,

    transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),

    transforms.Compose([
        transforms.Resize((IMG_SIZE + 48, IMG_SIZE + 48)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),

    transforms.Compose([
        transforms.Resize((IMG_SIZE + 48, IMG_SIZE + 48)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]


@torch.no_grad()
def predict(model, file_ids: list[str], tta: bool = True) -> dict[str, int]:
    model.eval()

    tf_list = TTA_TF_LIST if tta else [EVAL_TF]
    all_probs = {}

    for tf in tf_list:
        ds = TestDataset(TEST_DIR, file_ids, transform=tf)
        loader = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
        )

        for imgs, fnames in tqdm(loader, leave=False, desc="predict"):
            imgs = imgs.to(device)
            probs = torch.softmax(model(imgs), dim=1).cpu().numpy()

            for fname, p in zip(fnames, probs):
                if fname not in all_probs:
                    all_probs[fname] = np.zeros(NUM_CLASSES, dtype=np.float64)

                all_probs[fname] += p

    return {fname: int(np.argmax(p)) for fname, p in all_probs.items()}

In [11]:
set_seed()

sample_df = pd.read_csv(SAMPLE_CSV)
all_ids = sorted(
    [p.name for p in TEST_DIR.glob("*.jpg")],
    key=lambda x: int(x.split(".")[0])
)

print(len(all_ids))

print(f"Test images for submission: {len(all_ids)}")

model = build_model().to(device)

n_all = sum(p.numel() for p in model.parameters())
print(f"Model: EfficientNet-B2 | {n_all:,} total params")

ds_aug = TrainDataset(TRAIN_DIR, transform=TRAIN_TF)
ds_cln = TrainDataset(TRAIN_DIR, transform=EVAL_TF)

train_idx, val_idx = stratified_split_indices(ds_aug.samples, val_per_class=1)

train_ds = Subset(ds_aug, train_idx)
val_ds = Subset(ds_cln, val_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
best_val_acc = 0.0

1036
Test images for submission: 1036
Model: EfficientNet-B2 | 20,884,660 total params
Train: 979 | Val: 100


In [12]:
freeze_backbone(model)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1: head only | {n_trainable:,} trainable params")

opt1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR,
    weight_decay=WEIGHT_DECAY,
)

sch1 = torch.optim.lr_scheduler.OneCycleLR(
    opt1,
    max_lr=PHASE1_LR,
    total_steps=PHASE1_EPOCHS * len(train_loader),
    pct_start=0.2,
)

for ep in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(
        model,
        train_loader,
        opt1,
        criterion,
        sch1,
        use_mixup=False,
    )

    v_loss, v_acc = evaluate(model, val_loader, criterion)

    print(
        f"Epoch {ep:2d}/{PHASE1_EPOCHS} | "
        f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
        f"val loss {v_loss:.3f} acc {v_acc:.3f}"
    )

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": ep,
                "val_acc": v_acc,
            },
            CKPT_PATH,
        )
        print(f"New best: {best_val_acc:.4f}")

Phase 1: head only | 707,172 trainable params


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch  1/5 | train loss 4.353 acc 0.072 | val loss 3.868 acc 0.130
New best: 0.1300


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch  2/5 | train loss 3.205 acc 0.218 | val loss 3.001 acc 0.280
New best: 0.2800


...

Epoch  3/5 | train loss 2.624 acc 0.340 | val loss 2.705 acc 0.330
New best: 0.3300


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch  4/5 | train loss 2.271 acc 0.458 | val loss 2.510 acc 0.380
New best: 0.3800


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch  5/5 | train loss 2.122 acc 0.507 | val loss 2.490 acc 0.370


In [13]:
unfreeze_all(model)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2: full fine-tune | {n_trainable:,} trainable params")

classifier_ids = {id(p) for p in model.classifier.parameters()}

backbone_params = [
    p for p in model.parameters()
    if id(p) not in classifier_ids
]

classifier_params = list(model.classifier.parameters())

opt2 = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": PHASE2_LR},
        {"params": classifier_params, "lr": PHASE2_LR * 3},
    ],
    weight_decay=WEIGHT_DECAY,
)

sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt2,
    T_max=PHASE2_EPOCHS,
    eta_min=8e-7,
)

PATIENCE = 10
epochs_without_improvement = 0

for ep in range(1, 46):
    tr_loss, tr_acc = train_one_epoch(
        model,
        train_loader,
        opt2,
        criterion,
        scheduler=None,
        use_mixup=False,
    )

    sch2.step()

    v_loss, v_acc = evaluate(model, val_loader, criterion)

    print(
        f"Epoch {ep:2d}/45 | "
        f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
        f"val loss {v_loss:.3f} acc {v_acc:.3f}"
    )

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": PHASE1_EPOCHS + ep,
                "val_acc": v_acc,
            },
            CKPT_PATH,
        )

        print(f"New best: {best_val_acc:.4f}")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without improvement.")
        break

print(f"Done. Best val acc: {best_val_acc:.4f}")

Phase 2: full fine-tune | 20,884,660 trainable params


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch  1/45 | train loss 1.936 acc 0.517 | val loss 1.915 acc 0.490
New best: 0.4900


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch  2/45 | train loss 1.441 acc 0.669 | val loss 1.594 acc 0.590
New best: 0.5900


...

Epoch 31/45 | train loss 0.301 acc 1.000 | val loss 0.858 acc 0.840


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch 32/45 | train loss 0.298 acc 1.000 | val loss 0.904 acc 0.830


train:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch 33/45 | train loss 0.300 acc 0.999 | val loss 0.916 acc 0.830
Early stopping after 10 epochs without improvement.
Done. Best val acc: 0.8400


In [14]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

print(f"Loaded checkpoint from epoch {ckpt['epoch']} with val_acc {ckpt['val_acc']:.4f}")

Loaded checkpoint from epoch 28 with val_acc 0.8400


In [15]:
predictions = predict(model, all_ids, tta=True)

submission_df = pd.DataFrame({
    "ID": all_ids,
    "Label": [predictions[fid] for fid in all_ids],
})

submission_df.to_csv("submission.csv", index=False)

print(submission_df.shape)
print(submission_df.head())
print(submission_df.tail())
print(submission_df["Label"].min(), submission_df["Label"].max())

predict:   0%|          | 0/87 [00:00<?, ?it/s]

predict:   0%|          | 0/87 [00:00<?, ?it/s]

predict:   0%|          | 0/87 [00:00<?, ?it/s]

predict:   0%|          | 0/87 [00:00<?, ?it/s]

(1036, 2)
      ID  Label
0  0.jpg     62
1  1.jpg     43
2  2.jpg     38
3  3.jpg     51
4  4.jpg     42
            ID  Label
1031  1031.jpg     31
1032  1032.jpg      2
1033  1033.jpg     12
1034  1034.jpg     94
1035  1035.jpg     59
0 99
